In [6]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as snb
import os 
import seaborn as sns
from scipy import stats
from itertools import combinations
from scipy.stats import bootstrap
data_filepath = "../data/interventions/"

In [11]:
data_intervention_A_df = pd.read_csv(data_filepath+'data_A=-2.csv').drop(columns=["Unnamed: 0"])
#data_intervention_B_df = pd.read_csv(data_filepath+'data_B=0.csv').drop(columns=["Unnamed: 0"])
#data_intervention_C_df = pd.read_csv(data_filepath+'data_C=0.csv').drop(columns=["Unnamed: 0"])
#data_intervention_D_df = pd.read_csv(data_filepath+'data_D=0.csv').drop(columns=["Unnamed: 0"])
#data_intervention_H_df = pd.read_csv(data_filepath+'data_H=0.csv').drop(columns=["Unnamed: 0"])
data = pd.read_csv(data_filepath+'data.csv').drop(columns=["Unnamed: 0"])
datasets = {
    "data": data,
    "intervention_A": data_intervention_A_df,
    #"intervention_B": data_intervention_B_df,
    #"intervention_C": data_intervention_C_df,
    #"intervention_D": data_intervention_D_df,
    #"intervention_H": data_intervention_H_df,
}

data_var = ["A", "B", "C", "D", "E", "F"]

In [8]:
rng = np.random.default_rng(42)

bootstrap_results = {}

for name, df in datasets.items():         
    bootstrap_results[name] = {}
    for var in data_var:
        data = (df[var].values,)       
        res = bootstrap(
            data,
            np.std,
            confidence_level=0.90,
            n_resamples=200,
            random_state=rng
        )
      

        bootstrap_results[name][var] = res.bootstrap_distribution

print(len(bootstrap_results["data"]["A"]))

200


C:\Users\45512\anaconda3\Lib\site-packages\scipy\stats\_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
C:\Users\45512\AppData\Local\Temp\ipykernel_11956\4103174370.py:10: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  res = bootstrap(


In [13]:
def significant_difference(datasets: dict) -> pd.DataFrame:
    
    results = []
    
    for (name1, data_1), (name2, data_2) in combinations(datasets.items(), 2):
        for var in data_var:
            ks_result = stats.kstest(data_1[var], data_2[var])
            p_value = ks_result.pvalue
            results.append({
                "Dataset 1": name1,
                "Dataset 2": name2,
                "Variable": var,
                "KS Statistic": round(ks_result.statistic, 4),
                "P-Value": round(p_value, 4),
                "Identical Distribution": p_value >= 0.05,
                "mean-difference": np.abs(np.mean(data_1[var]) - np.mean(data_2[var])),
                "var-difference": np.abs(np.std(data_1[var]) - np.std(data_2[var]))
            })
    
    return pd.DataFrame(results)




results_df = significant_difference(datasets)
results_df

,Dataset 1,Dataset 2,Variable,KS Statistic,P-Value,Identical Distribution,mean-difference,var-difference
0,data,intervention_A,A,1.00,0.0000,False,10.738074,18.703372
1,data,intervention_A,B,0.18,0.0782,True,1.180929,0.485534
2,data,intervention_A,C,0.16,0.1548,True,0.236134,0.165871
3,data,intervention_A,D,0.09,0.8154,True,0.393297,0.498919
4,data,intervention_A,E,0.07,0.9684,True,0.034736,0.084555
5,data,intervention_A,F,0.09,0.8154,True,0.173499,0.054379


In [10]:
# Step 3+4: Run all pairwise KS tests and build edge candidates
intervention_datasets = {
    "A": data_intervention_A_df,
    #"B": data_intervention_B_df,
    #"C": data_intervention_C_df,
    #"D": data_intervention_D_df,
    # _H intentionally omitted — unobserved in practice
}

observed_vars = ["A", "B", "C", "D"]
results = []

for intervened_var, int_df in intervention_datasets.items():
    for var in observed_vars:
        if var == intervened_var:
            continue  # skip self-comparison
        ks = stats.kstest(data[var], int_df[var])
        results.append({
            "Intervened on": intervened_var,
            "Observed var": var,
            "KS statistic": round(ks.statistic, 4),
            "p-value": round(ks.pvalue, 4),
            "Distribution changed": ks.pvalue < 0.05
        })

ks_df = pd.DataFrame(results)

# Step 5: Print your guess
guess = [('A','B'), ('_H','A'), ('_H','D'), ('B','C'), ('C','D')]
print("Guess:", guess)
print("\nKS test summary:")
print(ks_df[ks_df["Distribution changed"]])

TypeError: tuple indices must be integers or slices, not str

In [ ]:
significant_difference(data, data_intervention_A_df)


In [ ]:
graphs = [data, data_intervention_A_df,] #data_intervention_B_df, data_intervention_C_df,data_intervention_D_df, data_intervention_H_df]
names = ["None", "A","B", "C", "D", "H"]
for i ,d in enumerate(graphs):
    print("\n\n graph number:", i, " with intervention on: ", names[i])
    print(d.describe())

In [ ]:
sns.pairplot(data=data)
